In [0]:
# ===================================================
# BLOCK 1 — AUDIT CONFIGURATION (PYTHON)
# ===================================================

"""
Define the governed source layers and production-day convention evaluated
before the SemiconPlus Gold dimensional model is rebuilt.
"""

from functools import reduce

from pyspark.sql import functions as F
from pyspark.sql import types as T

CATALOG = "semiconplus_portfolio"
SOURCE_SCHEMAS = ["bronze", "silver"]

LOCAL_TIMEZONE = "Asia/Taipei"
PRODUCTION_DAY_START_HOUR = 8

print(f"Catalog: {CATALOG}")
print(f"Source schemas: {SOURCE_SCHEMAS}")
print(f"Local timezone: {LOCAL_TIMEZONE}")
print(f"Production-day start hour: {PRODUCTION_DAY_START_HOUR:02d}:00")

In [0]:
# ===================================================
# BLOCK 2 — CATALOG AND SCHEMA INVENTORY (PYTHON)
# ===================================================

"""
Confirm that the required Unity Catalog source schemas exist before collecting
table-level metadata and five-year coverage evidence.
"""

catalogs = {
    row["catalog"]
    for row in spark.sql("SHOW CATALOGS").collect()
}

assert CATALOG in catalogs, f"Catalog does not exist: {CATALOG}"

available_schemas = {
    row["databaseName"]
    for row in spark.sql(f"SHOW SCHEMAS IN {CATALOG}").collect()
}

schema_results = [
    (schema_name, schema_name in available_schemas)
    for schema_name in SOURCE_SCHEMAS
]

schema_inventory_df = spark.createDataFrame(
    schema_results,
    ["schema_name", "schema_exists"],
)

display(schema_inventory_df)

missing_schemas = [
    schema_name
    for schema_name, schema_exists in schema_results
    if not schema_exists
]

assert not missing_schemas, (
    f"Required schemas do not exist: {missing_schemas}"
)

In [0]:
# ===================================================
# BLOCK 3 — SOURCE TABLE INVENTORY (PYTHON)
# ===================================================

"""
Inventory all Bronze and Silver tables and views that may supply the redesigned
dimensions, facts, governance outputs, and operational measures.
"""

inventory_records = []

for schema_name in SOURCE_SCHEMAS:
    objects = spark.sql(
        f"SHOW TABLES IN {CATALOG}.{schema_name}"
    ).collect()

    for source_object in objects:
        table_name = source_object["tableName"]
        qualified_name = f"{CATALOG}.{schema_name}.{table_name}"

        detail_row = spark.sql(
            f"DESCRIBE DETAIL {qualified_name}"
        ).first()

        inventory_records.append(
            (
                schema_name,
                table_name,
                qualified_name,
                detail_row["format"],
                detail_row["location"],
                str(detail_row["createdAt"]),
                str(detail_row["lastModified"]),
            )
        )

source_inventory_df = spark.createDataFrame(
    inventory_records,
    [
        "schema_name",
        "table_name",
        "qualified_name",
        "storage_format",
        "storage_location",
        "created_at",
        "last_modified_at",
    ],
)

display(
    source_inventory_df.orderBy(
        "schema_name",
        "table_name",
    )
)

print(f"Governed source objects found: {source_inventory_df.count():,}")

In [0]:
# ===================================================
# BLOCK 4 — SOURCE ROW-COUNT BASELINE (PYTHON)
# ===================================================

"""
Capture the current row count for every governed source table so future Gold
transformations can be reconciled against a fixed Day 1 baseline.
"""

row_count_results = []

for source_object in source_inventory_df.collect():
    qualified_name = source_object["qualified_name"]

    try:
        row_count = spark.table(qualified_name).count()
        status = "PASSED"
        error_message = None
    except Exception as error:
        row_count = None
        status = "FAILED"
        error_message = str(error)[:500]

    row_count_results.append(
        (
            source_object["schema_name"],
            source_object["table_name"],
            qualified_name,
            row_count,
            status,
            error_message,
        )
    )

row_count_df = spark.createDataFrame(
    row_count_results,
    schema="""
        schema_name STRING,
        table_name STRING,
        qualified_name STRING,
        row_count LONG,
        audit_status STRING,
        error_message STRING
    """,
)

display(
    row_count_df.orderBy(
        "schema_name",
        "table_name",
    )
)

failed_row_count_tables = (
    row_count_df
    .filter(F.col("audit_status") == "FAILED")
    .count()
)

assert failed_row_count_tables == 0, (
    f"Row-count audit failed for {failed_row_count_tables} objects."
)

In [0]:
# ===================================================
# BLOCK 5 — BUSINESS-FIELD INVENTORY (PYTHON)
# ===================================================

"""
Search source schemas for the fields required to model sites, products, lots,
yield, retest, equipment, downtime, alarms, and pipeline controls.
"""

BUSINESS_FIELD_PATTERNS = {
    "SITE": ["site"],
    "PRODUCT_GROUP": ["product_group", "business_group"],
    "DEVICE": ["device", "product"],
    "EQUIPMENT": ["equipment", "tester", "handler"],
    "SOURCE_LOT": ["lot_id", "lot"],
    "MOTHER_LOT": ["mother_lot", "parent_lot"],
    "SUBLOT": ["sublot", "sub_lot"],
    "INPUT_QUANTITY": [
        "input_quantity",
        "input_qty",
        "units_started",
        "tested_quantity",
    ],
    "FIRST_PASS_GOOD": [
        "first_pass_good",
        "first_pass_pass",
        "passed",
        "pass_count",
    ],
    "RETEST_INPUT": [
        "retest_input",
        "retest_quantity",
        "retest_qty",
    ],
    "RETEST_GOOD": [
        "retest_good",
        "retest_pass",
        "recovered",
    ],
    "DEFECT_ERROR": [
        "defect",
        "error_code",
        "alarm_code",
    ],
    "EVENT_TIMESTAMP": [
        "timestamp",
        "event_time",
        "start_time",
        "end_time",
    ],
    "DURATION": ["duration", "seconds"],
    "EQUIPMENT_STATE": ["event_type", "equipment_state", "status"],
    "PIPELINE_METADATA": [
        "pipeline_run",
        "ingested_at",
        "processed_at",
        "source_file",
    ],
}

field_records = []

for source_object in source_inventory_df.collect():
    qualified_name = source_object["qualified_name"]
    source_fields = spark.table(qualified_name).schema.fields

    for source_field in source_fields:
        normalized_column = source_field.name.lower()

        matched_categories = [
            category
            for category, patterns in BUSINESS_FIELD_PATTERNS.items()
            if any(pattern in normalized_column for pattern in patterns)
        ]

        field_records.append(
            (
                source_object["schema_name"],
                source_object["table_name"],
                qualified_name,
                source_field.name,
                source_field.dataType.simpleString(),
                source_field.nullable,
                ",".join(matched_categories)
                if matched_categories else "UNCLASSIFIED",
            )
        )

field_inventory_df = spark.createDataFrame(
    field_records,
    [
        "schema_name",
        "table_name",
        "qualified_name",
        "column_name",
        "data_type",
        "nullable",
        "business_categories",
    ],
)

display(
    field_inventory_df
    .filter(F.col("business_categories") != "UNCLASSIFIED")
    .orderBy(
        "business_categories",
        "schema_name",
        "table_name",
        "column_name",
    )
)

In [0]:
# ===================================================
# BLOCK 6 — DATE-COLUMN DISCOVERY (PYTHON)
# ===================================================

"""
Identify date and timestamp candidates that can establish historical coverage
and support UTC, local-time, and configurable production-day derivations.
"""

date_column_records = []

for source_object in source_inventory_df.collect():
    qualified_name = source_object["qualified_name"]

    for source_field in spark.table(qualified_name).schema.fields:
        column_name = source_field.name
        column_type = source_field.dataType.simpleString().lower()
        normalized_name = column_name.lower()

        is_date_candidate = (
            isinstance(
                source_field.dataType,
                (T.DateType, T.TimestampType),
            )
            or "date" in normalized_name
            or "time" in normalized_name
            or "timestamp" in normalized_name
        )

        if is_date_candidate:
            date_column_records.append(
                (
                    source_object["schema_name"],
                    source_object["table_name"],
                    qualified_name,
                    column_name,
                    column_type,
                )
            )

date_column_inventory_df = spark.createDataFrame(
    date_column_records,
    [
        "schema_name",
        "table_name",
        "qualified_name",
        "date_column",
        "source_data_type",
    ],
)

display(
    date_column_inventory_df.orderBy(
        "schema_name",
        "table_name",
        "date_column",
    )
)

In [0]:
# ===================================================
# BLOCK 7 — FIVE-YEAR BUSINESS-DATE COVERAGE (PYTHON)
# ===================================================

"""
Measure the usable range of every discovered date candidate. Results distinguish
business history from ingestion and file-modification metadata.
"""

coverage_results = []

for date_column in date_column_inventory_df.collect():
    qualified_name = date_column["qualified_name"]
    column_name = date_column["date_column"]

    try:
        coverage = (
            spark.table(qualified_name)
            .select(
                F.expr(
                    f"try_cast(`{column_name}` AS timestamp)"
                ).alias("audited_timestamp")
            )
            .agg(
                F.min("audited_timestamp").alias("minimum_timestamp"),
                F.max("audited_timestamp").alias("maximum_timestamp"),
                F.count("audited_timestamp").alias("non_null_count"),
            )
            .first()
        )

        minimum_timestamp = coverage["minimum_timestamp"]
        maximum_timestamp = coverage["maximum_timestamp"]

        if minimum_timestamp and maximum_timestamp:
            span_days = (
                maximum_timestamp.date()
                - minimum_timestamp.date()
            ).days
        else:
            span_days = None

        status = "PASSED"
        error_message = None

    except Exception as error:
        minimum_timestamp = None
        maximum_timestamp = None
        span_days = None
        coverage = {"non_null_count": None}
        status = "FAILED"
        error_message = str(error)[:500]

    coverage_results.append(
        (
            date_column["schema_name"],
            date_column["table_name"],
            qualified_name,
            column_name,
            str(minimum_timestamp) if minimum_timestamp else None,
            str(maximum_timestamp) if maximum_timestamp else None,
            span_days,
            coverage["non_null_count"],
            status,
            error_message,
        )
    )

date_coverage_df = spark.createDataFrame(
    coverage_results,
    schema="""
        schema_name STRING,
        table_name STRING,
        qualified_name STRING,
        date_column STRING,
        minimum_timestamp STRING,
        maximum_timestamp STRING,
        span_days LONG,
        non_null_count LONG,
        audit_status STRING,
        error_message STRING
    """,
)

display(
    date_coverage_df.orderBy(
        F.desc_nulls_last("span_days")
    )
)

In [0]:
# ===================================================
# BLOCK 8 — CRITICAL-FIELD GAP REPORT (PYTHON)
# ===================================================

"""
Summarize whether source fields exist for each critical modeling requirement.
A discovered name is evidence of availability, not proof of business meaning.
"""

CRITICAL_REQUIREMENTS = [
    "SITE",
    "PRODUCT_GROUP",
    "DEVICE",
    "EQUIPMENT",
    "SOURCE_LOT",
    "MOTHER_LOT",
    "SUBLOT",
    "INPUT_QUANTITY",
    "FIRST_PASS_GOOD",
    "RETEST_INPUT",
    "RETEST_GOOD",
    "DEFECT_ERROR",
    "EVENT_TIMESTAMP",
    "DURATION",
    "EQUIPMENT_STATE",
    "PIPELINE_METADATA",
]

requirement_results = []

for requirement in CRITICAL_REQUIREMENTS:
    matches = (
        field_inventory_df
        .filter(
            F.array_contains(
                F.split("business_categories", ","),
                requirement,
            )
        )
        .select(
            "qualified_name",
            "column_name",
            "data_type",
        )
        .collect()
    )

    evidence = "; ".join(
        f"{row['qualified_name']}.{row['column_name']} "
        f"({row['data_type']})"
        for row in matches
    )

    requirement_results.append(
        (
            requirement,
            len(matches),
            "FOUND_REVIEW_REQUIRED" if matches else "MISSING",
            evidence if evidence else None,
        )
    )

critical_gap_df = spark.createDataFrame(
    requirement_results,
    [
        "requirement",
        "matching_field_count",
        "availability_status",
        "source_evidence",
    ],
)

display(critical_gap_df.orderBy("requirement"))

In [0]:
# ===================================================
# BLOCK 9 — PRODUCTION-DAY DERIVATION TEST (PYTHON)
# ===================================================

"""
Validate the configurable UTC-to-Taipei production-day calculation before the
same convention is applied across dated Gold facts.
"""

production_day_test_df = spark.createDataFrame(
    [
        ("2026-01-01T23:59:59Z",),
        ("2026-01-02T00:00:00Z",),
        ("2026-01-02T07:59:59Z",),
        ("2026-01-02T08:00:00Z",),
    ],
    ["source_timestamp_utc"],
)

production_day_test_df = (
    production_day_test_df
    .withColumn(
        "event_timestamp_utc",
        F.to_timestamp("source_timestamp_utc"),
    )
    .withColumn(
        "event_timestamp_local",
        F.from_utc_timestamp(
            "event_timestamp_utc",
            LOCAL_TIMEZONE,
        ),
    )
    .withColumn(
        "production_date",
        F.to_date(
            F.expr(
                "event_timestamp_local "
                f"- INTERVAL {PRODUCTION_DAY_START_HOUR} HOURS"
            )
        ),
    )
)

display(production_day_test_df)

assert (
    production_day_test_df
    .filter(F.col("production_date").isNull())
    .count()
    == 0
)

print("Parameterized production-day derivation passed.")

In [0]:
# ===================================================
# BLOCK 10 — EXACT MODELING-SOURCE SCHEMAS (PYTHON)
# ===================================================

"""
Display the complete schemas of the authoritative transactional sources used
to determine the supported yield, retest, lot, and OEE contracts.
"""

AUTHORITATIVE_SOURCES = [
    "semiconplus_portfolio.silver.production_lots",
    "semiconplus_portfolio.silver.unit_test_results",
    "semiconplus_portfolio.silver.equipment_events",
    "semiconplus_portfolio.silver.tester_logs",
    "semiconplus_portfolio.silver.devices",
    "semiconplus_portfolio.silver.equipment",
]

schema_records = []

for table_name in AUTHORITATIVE_SOURCES:
    print("=" * 100)
    print(table_name)
    print("=" * 100)

    source_df = spark.table(table_name)
    source_df.printSchema()

    for field in source_df.schema.fields:
        schema_records.append(
            (
                table_name,
                field.name,
                field.dataType.simpleString(),
                field.nullable,
            )
        )

exact_schema_df = spark.createDataFrame(
    schema_records,
    [
        "qualified_table_name",
        "column_name",
        "data_type",
        "nullable",
    ],
)

display(
    exact_schema_df.orderBy(
        "qualified_table_name",
        "column_name",
    )
)

In [0]:
# ===================================================
# BLOCK 11 — PRODUCTION-LOT QUANTITY PROFILE (PYTHON)
# ===================================================

"""
Inspect production-lot quantity fields and mathematical consistency to identify
the authoritative inputs for FPY, final yield, and rejected-lot calculations.
"""

production_lots_df = spark.table(
    "semiconplus_portfolio.silver.production_lots"
)

quantity_columns = [
    field.name
    for field in production_lots_df.schema.fields
    if any(
        keyword in field.name.lower()
        for keyword in [
            "quantity",
            "qty",
            "input",
            "started",
            "passed",
            "failed",
            "good",
            "retest",
        ]
    )
]

print("Production-lot quantity candidates:")
for column_name in quantity_columns:
    print(f"- {column_name}")

production_lots_df.select(
    "lot_id",
    "production_date",
    "device_id",
    "product_group_id",
    "site_id",
    "equipment_id",
    *quantity_columns,
).show(
    n=20,
    truncate=False,
)

In [0]:
# ===================================================
# BLOCK 12 — PRODUCTION-LOT QUANTITY STATISTICS (PYTHON)
# ===================================================

"""
Measure nulls, minimums, maximums, and totals for candidate quantity columns
before assigning them to yield numerator and denominator contracts.
"""

numeric_quantity_columns = [
    column_name
    for column_name in quantity_columns
    if isinstance(
        production_lots_df.schema[column_name].dataType,
        (
            T.ByteType,
            T.ShortType,
            T.IntegerType,
            T.LongType,
            T.FloatType,
            T.DoubleType,
            T.DecimalType,
        ),
    )
]

quantity_statistics = []

for column_name in numeric_quantity_columns:
    metrics = production_lots_df.agg(
        F.count("*").alias("source_rows"),
        F.count(F.col(column_name)).alias("non_null_rows"),
        F.sum(
            F.when(F.col(column_name).isNull(), 1).otherwise(0)
        ).alias("null_rows"),
        F.min(column_name).alias("minimum_value"),
        F.max(column_name).alias("maximum_value"),
        F.sum(column_name).alias("total_value"),
    ).first()

    quantity_statistics.append(
        (
            column_name,
            metrics["source_rows"],
            metrics["non_null_rows"],
            metrics["null_rows"],
            float(metrics["minimum_value"])
            if metrics["minimum_value"] is not None else None,
            float(metrics["maximum_value"])
            if metrics["maximum_value"] is not None else None,
            float(metrics["total_value"])
            if metrics["total_value"] is not None else None,
        )
    )

quantity_statistics_df = spark.createDataFrame(
    quantity_statistics,
    schema="""
        column_name STRING,
        source_rows LONG,
        non_null_rows LONG,
        null_rows LONG,
        minimum_value DOUBLE,
        maximum_value DOUBLE,
        total_value DOUBLE
    """,
)

display(quantity_statistics_df.orderBy("column_name"))

In [0]:
# ===================================================
# BLOCK 13 — UNIT-TEST RESULT CONTRACT PROFILE (PYTHON)
# ===================================================

"""
Profile unit-level results to determine whether retest attempts are explicitly
represented or whether the existing source contains only final unit outcomes.
"""

unit_results_df = spark.table(
    "semiconplus_portfolio.silver.unit_test_results"
)

unit_results_df.printSchema()

display(
    unit_results_df
    .select(
        *[
            column_name
            for column_name in [
                "test_result_id",
                "event_timestamp_utc",
                "lot_id",
                "unit_sequence",
                "device_id",
                "product_group_id",
                "site_id",
                "equipment_id",
                "test_status",
                "defect_code",
                "test_time_seconds",
            ]
            if column_name in unit_results_df.columns
        ]
    )
    .limit(25)
)

for category_column in [
    "test_status",
    "defect_code",
]:
    if category_column in unit_results_df.columns:
        print(f"Distribution: {category_column}")

        display(
            unit_results_df
            .groupBy(category_column)
            .count()
            .orderBy(F.desc("count"))
        )

In [0]:
# ===================================================
# BLOCK 14 — REPEATED UNIT-ATTEMPT ANALYSIS (PYTHON)
# ===================================================

"""
Test whether a stable unit identifier appears in multiple chronological records.
Repeated records can support retest modeling only when the same physical unit
and ordered test attempt can be identified reliably.
"""

UNIT_KEY_CANDIDATES = [
    column_name
    for column_name in [
        "unit_id",
        "unit_serial_number",
        "serial_number",
        "barcode",
        "lot_id",
        "unit_sequence",
    ]
    if column_name in unit_results_df.columns
]

print(f"Available unit-key candidates: {UNIT_KEY_CANDIDATES}")

if "lot_id" in unit_results_df.columns and "unit_sequence" in unit_results_df.columns:
    repeated_unit_attempts_df = (
        unit_results_df
        .groupBy(
            "lot_id",
            "unit_sequence",
        )
        .agg(
            F.count("*").alias("result_record_count"),
            F.countDistinct("test_status").alias(
                "distinct_status_count"
            ),
            F.min("event_timestamp_utc").alias(
                "first_test_timestamp_utc"
            ),
            F.max("event_timestamp_utc").alias(
                "last_test_timestamp_utc"
            ),
        )
        .filter(F.col("result_record_count") > 1)
    )

    repeated_unit_count = repeated_unit_attempts_df.count()

    print(
        "Repeated lot/unit-sequence combinations: "
        f"{repeated_unit_count:,}"
    )

    display(
        repeated_unit_attempts_df
        .orderBy(F.desc("result_record_count"))
        .limit(25)
    )
else:
    print(
        "No supported lot/unit composite identifier is available "
        "for repeated-attempt analysis."
    )

In [0]:
# ===================================================
# BLOCK 15 — LOT-HIERARCHY ASSESSMENT (PYTHON)
# ===================================================

"""
Confirm whether source mother-lot and sublot identifiers exist and document
whether deterministic simulated hierarchy mappings are required.
"""

lot_hierarchy_candidates = [
    column_name
    for column_name in production_lots_df.columns
    if any(
        keyword in column_name.lower()
        for keyword in [
            "mother",
            "parent",
            "sublot",
            "sub_lot",
            "wafer",
            "batch",
        ]
    )
]

print(f"Lot hierarchy candidates: {lot_hierarchy_candidates}")

if lot_hierarchy_candidates:
    display(
        production_lots_df
        .select(
            "lot_id",
            *lot_hierarchy_candidates,
        )
        .limit(25)
    )
else:
    print(
        "No source mother-lot, sublot, wafer-batch, or parent-lot "
        "field was found. A persisted deterministic simulated mapping "
        "will be required if this analytics scope is retained."
    )

In [0]:
# ===================================================
# BLOCK 16 — EQUIPMENT-EVENT CLASSIFICATION PROFILE (PYTHON)
# ===================================================

"""
Profile equipment states, duration ranges, and alarms to confirm which raw
event categories can support planned downtime, unplanned downtime, short stops,
availability, and utilization calculations.
"""

equipment_events_df = spark.table(
    "semiconplus_portfolio.silver.equipment_events"
)

display(
    equipment_events_df
    .groupBy("event_type")
    .agg(
        F.count("*").alias("event_count"),
        F.sum("duration_seconds").alias("total_duration_seconds"),
        F.min("duration_seconds").alias("minimum_duration_seconds"),
        F.max("duration_seconds").alias("maximum_duration_seconds"),
        F.avg("duration_seconds").alias("average_duration_seconds"),
        F.sum(
            F.when(F.col("alarm_code").isNotNull(), 1).otherwise(0)
        ).alias("events_with_alarm"),
    )
    .orderBy(F.desc("total_duration_seconds"))
)

In [0]:
# ===================================================
# BLOCK 17 — OEE TIME-COMPONENT FEASIBILITY (PYTHON)
# ===================================================

"""
Aggregate event durations by equipment and UTC date to determine whether source
event intervals provide explainable scheduled, planned, operating, downtime,
and short-stop components for OEE.
"""

oee_feasibility_df = (
    equipment_events_df
    .withColumn(
        "event_date_utc",
        F.to_date("event_timestamp_utc"),
    )
    .groupBy(
        "event_date_utc",
        "site_id",
        "equipment_id",
    )
    .agg(
        F.sum("duration_seconds").alias(
            "observed_event_seconds"
        ),
        F.sum(
            F.when(
                F.col("event_type") == "RUN",
                F.col("duration_seconds"),
            ).otherwise(0)
        ).alias("run_seconds"),
        F.sum(
            F.when(
                F.col("event_type").isin(
                    "MAINTENANCE",
                    "PLANNED_DOWNTIME",
                    "SETUP",
                ),
                F.col("duration_seconds"),
            ).otherwise(0)
        ).alias("planned_candidate_seconds"),
        F.sum(
            F.when(
                (F.col("event_type").isin(
                    "ALARM",
                    "UNPLANNED_DOWNTIME",
                ))
                & (F.col("duration_seconds") > 300),
                F.col("duration_seconds"),
            ).otherwise(0)
        ).alias("unplanned_downtime_seconds"),
        F.sum(
            F.when(
                (F.col("event_type").isin(
                    "IDLE",
                    "ALARM",
                    "UNPLANNED_DOWNTIME",
                ))
                & (F.col("duration_seconds") <= 300),
                F.col("duration_seconds"),
            ).otherwise(0)
        ).alias("short_stop_seconds"),
    )
)

display(
    oee_feasibility_df
    .agg(
        F.min("observed_event_seconds").alias(
            "minimum_daily_observed_seconds"
        ),
        F.max("observed_event_seconds").alias(
            "maximum_daily_observed_seconds"
        ),
        F.avg("observed_event_seconds").alias(
            "average_daily_observed_seconds"
        ),
        F.sum(
            F.when(
                F.col("observed_event_seconds") > 86400,
                1,
            ).otherwise(0)
        ).alias("equipment_days_above_86400_seconds"),
    )
)

In [0]:
# ===================================================
# BLOCK 18 — SILVER REJECTION RECONCILIATION (PYTHON)
# ===================================================

"""
Record Bronze-to-Silver count differences that must reconcile to documented
quality, duplicate, late-arrival, or quarantine outcomes.
"""

reconciliation_records = [
    ("production_lots", 18126, 18124, 2),
    ("equipment_events", 255640, 255639, 1),
    ("unit_test_results", 181250, 181239, 11),
    ("streaming_test_results", 5001, 4956, 45),
    ("tester_logs", 18125, 0, 18125),
]

reconciliation_df = spark.createDataFrame(
    reconciliation_records,
    [
        "dataset_name",
        "bronze_rows",
        "silver_rows",
        "difference_rows",
    ],
)

display(reconciliation_df)

In [0]:
# ===================================================
# BLOCK 19 — LOCATE THE DATABRICKS GIT REPOSITORY (PYTHON)
# ===================================================

"""
Resolve the repository root without assuming that every Databricks workspace
exposes Git folders through the same /Workspace/Repos or /Repos prefix.
"""

from pathlib import Path

REPOSITORY_NAME = "semiconplus-manufacturing-lakehouse"
REPOSITORY_OWNER = "bugastokenpatrick@gmail.com"

candidate_repo_roots = [
    Path(
        f"/Workspace/Repos/{REPOSITORY_OWNER}/{REPOSITORY_NAME}"
    ),
    Path(
        f"/Repos/{REPOSITORY_OWNER}/{REPOSITORY_NAME}"
    ),
]

existing_repo_roots = [
    candidate
    for candidate in candidate_repo_roots
    if candidate.exists() and candidate.is_dir()
]

assert existing_repo_roots, (
    "Repository folder was not found. Open the Git folder, copy its exact "
    "workspace path, and add it to candidate_repo_roots."
)

REPO_ROOT = existing_repo_roots[0]

print(f"Resolved repository root: {REPO_ROOT}")


In [0]:
# ===================================================
# BLOCK 19A — CORRECT REPOSITORY AUDIT RULES (PYTHON)
# ===================================================

"""
Include Databricks notebook source and recognize compound latest/previous
identifiers such as latest_file_name and previous_run_date.
"""

import re

SEARCHABLE_SUFFIXES = {
    ".py",
    ".sql",
    ".ipynb",
    ".yml",
    ".yaml",
    ".json",
}

SEARCH_RULES = {
    "CURRENT_DATE_FILTER": (
        r"current_date\s*\(|current_timestamp\s*\(|"
        r"date_sub\s*\(|date_add\s*\("
    ),
    "LATEST_ONLY_LOGIC": (
        r"\blatest(?:[_\s-][a-z0-9]+)*\b|"
        r"\byesterday\b|"
        r"\bprevious(?:[_\s-][a-z0-9]+)*\b|"
        r"row_number\s*\(.*order\s+by.*desc"
    ),
    "LIMITED_READ": (
        r"\.limit\s*\(|\bLIMIT\s+\d+\b"
    ),
    "FIXED_DATE_LITERAL": (
        r"\b20\d{2}-\d{2}-\d{2}\b"
    ),
    "OVERWRITE_WRITE": (
        r"mode\s*\(\s*[\"']overwrite[\"']\s*\)|"
        r"INSERT\s+OVERWRITE|"
        r"CREATE\s+OR\s+REPLACE\s+TABLE"
    ),
    "DESTRUCTIVE_SQL": (
        r"\bTRUNCATE\s+TABLE\b|"
        r"\bDELETE\s+FROM\b|"
        r"\bDROP\s+TABLE\b|"
        r"\bDROP\s+SCHEMA\b"
    ),
}

compiled_rules = {
    rule_name: re.compile(pattern, flags=re.IGNORECASE)
    for rule_name, pattern in SEARCH_RULES.items()
}

print("Repository audit rules corrected.")
print(f"Searchable suffixes: {sorted(SEARCHABLE_SUFFIXES)}")

In [0]:
# ===================================================
# BLOCK 20 — AUDIT HISTORICAL-DATA RESTRICTIONS (PYTHON)
# ===================================================

"""
Search repository code for date filters, limited reads and destructive write
patterns that may silently remove historical reporting data. Matches require
manual review; a match is evidence to inspect, not automatically a defect.
"""

import re

SEARCHABLE_SUFFIXES = {
    ".py",
    ".sql",
    ".ipynb",
    ".yml",
    ".yaml",
    ".json",
}

EXCLUDED_DIRECTORIES = {
    ".git",
    ".pytest_cache",
    "__pycache__",
    ".databricks",
}

SEARCH_RULES = {
    "CURRENT_DATE_FILTER": (
        r"current_date\s*\(|current_timestamp\s*\(|"
        r"date_sub\s*\(|date_add\s*\("
    ),
    "LATEST_ONLY_LOGIC": (
        r"\blatest[_a-z0-9]*|"
        r"\byesterday\b|"
        r"\bprevious[_a-z0-9]*|"
        r"row_number\s*\(.*order\s+by.*desc"
    ),
    "LIMITED_READ": r"\.limit\s*\(|\bLIMIT\s+\d+\b",
    "FIXED_DATE_LITERAL": r"\b20\d{2}-\d{2}-\d{2}\b",
    "OVERWRITE_WRITE": (
        r"mode\s*\(\s*[\"']overwrite[\"']\s*\)|"
        r"INSERT\s+OVERWRITE|CREATE\s+OR\s+REPLACE\s+TABLE"
    ),
    "DESTRUCTIVE_SQL": (
        r"\bTRUNCATE\s+TABLE\b|\bDELETE\s+FROM\b|"
        r"\bDROP\s+TABLE\b|\bDROP\s+SCHEMA\b"
    ),
}

compiled_rules = {
    rule_name: re.compile(pattern, flags=re.IGNORECASE)
    for rule_name, pattern in SEARCH_RULES.items()
}

restriction_matches = []

for file_path in sorted(REPO_ROOT.rglob("*")):
    if not file_path.is_file():
        continue
    if file_path.suffix.lower() not in SEARCHABLE_SUFFIXES:
        continue
    if any(part in EXCLUDED_DIRECTORIES for part in file_path.parts):
        continue

    try:
        file_lines = file_path.read_text(
            encoding="utf-8",
            errors="replace",
        ).splitlines()
    except OSError as error:
        restriction_matches.append(
            (
                str(file_path.relative_to(REPO_ROOT)),
                None,
                "FILE_READ_ERROR",
                str(error),
            )
        )
        continue

    for line_number, line_text in enumerate(file_lines, start=1):
        stripped_line = line_text.strip()

        if not stripped_line:
            continue

        for rule_name, compiled_rule in compiled_rules.items():
            if compiled_rule.search(stripped_line):
                restriction_matches.append(
                    (
                        str(file_path.relative_to(REPO_ROOT)),
                        line_number,
                        rule_name,
                        stripped_line[:500],
                    )
                )

restriction_schema = (
    "relative_file_path string, line_number int, "
    "restriction_type string, matched_line string"
)

restriction_df = spark.createDataFrame(
    restriction_matches,
    schema=restriction_schema,
)

print(f"Repository files reviewed: {REPO_ROOT}")
print(f"Potential restriction matches: {restriction_df.count():,}")

display(
    restriction_df.orderBy(
        "relative_file_path",
        "line_number",
        "restriction_type",
    )
)


In [0]:
# ===================================================
# BLOCK 20A — VERIFY REPOSITORY SCAN COVERAGE (PYTHON)
# ===================================================

"""
Confirm that the historical-restriction audit actually inspected repository
source files before accepting a zero-match result.
"""

from collections import Counter

all_repo_files = sorted(
    path
    for path in REPO_ROOT.rglob("*")
    if path.is_file()
    and not any(
        excluded_directory in path.parts
        for excluded_directory in EXCLUDED_DIRECTORIES
    )
)

searchable_repo_files = [
    path
    for path in all_repo_files
    if path.suffix.lower() in SEARCHABLE_SUFFIXES
]

suffix_counts = Counter(
    path.suffix.lower() or "<NO_SUFFIX>"
    for path in all_repo_files
)

print(f"All repository files discovered: {len(all_repo_files):,}")
print(f"Searchable source files discovered: {len(searchable_repo_files):,}")
print(f"File suffix counts: {dict(sorted(suffix_counts.items()))}")

assert searchable_repo_files, (
    "The repository exists, but no searchable source files were found. "
    "The zero-match restriction result is therefore not valid."
)

display(
    spark.createDataFrame(
        [
            (
                str(path.relative_to(REPO_ROOT)),
                path.suffix.lower(),
                path.stat().st_size,
            )
            for path in searchable_repo_files
        ],
        [
            "relative_file_path",
            "file_suffix",
            "file_size_bytes",
        ],
    ).orderBy("relative_file_path")
)

print("Repository scan coverage verified.")

In [0]:
# ===================================================
# BLOCK 20B — VERIFY RESTRICTION RULE PATTERNS (PYTHON)
# ===================================================

"""
Prove that every restriction rule recognizes a controlled example.
"""

controlled_examples = {
    "CURRENT_DATE_FILTER": (
        "WHERE production_date = current_date()"
    ),
    "LATEST_ONLY_LOGIC": (
        "latest_file_name = source_file_name"
    ),
    "LIMITED_READ": (
        "source_df.limit(10)"
    ),
    "FIXED_DATE_LITERAL": (
        "start_date = '2025-01-01'"
    ),
    "OVERWRITE_WRITE": (
        'target_df.write.mode("overwrite").saveAsTable(table)'
    ),
    "DESTRUCTIVE_SQL": (
        "DELETE FROM gold.example_table"
    ),
}

rule_test_results = []

for rule_name, example_text in controlled_examples.items():
    pattern_matched = bool(
        compiled_rules[rule_name].search(example_text)
    )

    rule_test_results.append(
        (
            rule_name,
            example_text,
            pattern_matched,
        )
    )

display(
    spark.createDataFrame(
        rule_test_results,
        [
            "restriction_type",
            "controlled_example",
            "pattern_matched",
        ],
    )
)

failed_rule_tests = [
    rule_name
    for rule_name, _, pattern_matched in rule_test_results
    if not pattern_matched
]

assert not failed_rule_tests, (
    f"Restriction scanner rules failed: {failed_rule_tests}"
)

print("Historical-restriction scanner self-test passed.")

In [0]:
# ===================================================
# BLOCK 21 — SUMMARIZE HISTORICAL RESTRICTION TYPES (PYTHON)
# ===================================================

"""
Summarize matches so potentially destructive or history-limiting patterns are
reviewed before informational LIMIT clauses used only for display samples.
"""

from pyspark.sql import functions as F

restriction_summary_df = (
    restriction_df
    .groupBy("restriction_type")
    .agg(
        F.count("*").alias("match_count"),
        F.countDistinct("relative_file_path").alias("affected_file_count"),
    )
    .orderBy("restriction_type")
)

display(restriction_summary_df)

high_risk_matches = restriction_df.filter(
    F.col("restriction_type").isin(
        "OVERWRITE_WRITE",
        "DESTRUCTIVE_SQL",
    )
)

print(f"High-risk matches requiring manual review: {high_risk_matches.count():,}")
display(high_risk_matches.orderBy("relative_file_path", "line_number"))

print(
    "Review rule: retain LIMIT only for display/testing, retain overwrite only "
    "when the entire target is intentionally rebuilt from complete history, "
    "and remove any production filter that restricts Gold to recent dates."
)

In [0]:
# ===================================================
# BLOCK 22 — DAY 1 SOURCE ACCEPTANCE SUMMARY (PYTHON)
# ===================================================

"""
Publish the locked Day 1 conclusions. This block does not claim that synthetic
retest or OEE data already exists; it records that explicit source creation is
required on Days 3 and 4.
"""

day_1_acceptance = [
    (
        "HISTORICAL_COVERAGE",
        "PASSED",
        "Business data covers 2021-01-01 through 2025-12-31.",
    ),
    (
        "PRODUCTION_QUANTITIES",
        "PASSED",
        "quantity_started = quantity_passed + quantity_failed.",
    ),
    (
        "TEST_BATCH_HIERARCHY",
        "LOCKED_DERIVATION",
        "Use persisted simulated test_batch_id and test_lot_id mappings.",
    ),
    (
        "RETEST_SOURCE",
        "SYNTHETIC_SOURCE_REQUIRED",
        "Unit samples contain no repeated attempts; retest will be generated.",
    ),
    (
        "OEE_SOURCE",
        "SYNTHETIC_SOURCE_REQUIRED",
        "Observed events lack a complete operating schedule.",
    ),
    (
        "OBSERVED_EQUIPMENT_EVENTS",
        "PASSED_WITH_LIMITATION",
        "Suitable for event investigation, not standalone standard OEE.",
    ),
    (
        "PRODUCTION_DAY",
        "PASSED",
        "Asia/Taipei production day starts at 08:00 local time.",
    ),
    (
        "TESTER_LOGS_SILVER",
        "KNOWN_LIMITATION",
        "Silver tester_logs contains zero rows and is excluded from Gold dependencies.",
    ),
]

day_1_acceptance_df = spark.createDataFrame(
    day_1_acceptance,
    ["decision_area", "acceptance_status", "decision_evidence"],
)

display(day_1_acceptance_df)

blocking_statuses = {
    "FAILED",
    "UNRESOLVED",
}

blocking_count = day_1_acceptance_df.filter(
    F.col("acceptance_status").isin(*blocking_statuses)
).count()

assert blocking_count == 0, (
    f"Day 1 has {blocking_count} unresolved blocking decisions."
)

print("DAY 1 TECHNICAL AUDIT: PASSED")
print("Complete the documentation and manual restriction review before Day 2.")